In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from torch.utils.data import Dataset, DataLoader
import torch
print(torch.__version__)
print(torch.cuda.is_available())      # 应该是 True
print(torch.cuda.get_device_name(0))  # 应该显示 RTX 5060

2.10.0+cu128
True
NVIDIA GeForce RTX 5070 Laptop GPU


In [2]:
CSV_PATH=r"C:\Users\xhu70\Projects\twel_data_collection\data\sensor_data.csv"
WINDOW_SIZE=360 #用过去的360步（1小时）预测
PREDICTION_AHEAD=180
TARGET_COL="temperature_c"
BATCH_SIZE=64
HIDDEN_SIZE=64
NUM_LAYERS=2
EPOCHS=30
LR=0.001

In [3]:
device=torch.device("cuda" if torch.cuda.is_available() else "CPU")
print(device)

cuda


In [4]:
df=pd.read_csv(CSV_PATH,parse_dates=["timestamp"])
df=df.sort_values("timestamp").reset_index(drop=True)


In [5]:
print(f"数据总行数: {len(df)}")
print(f"时间范围: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"缺失值:\n{df.isnull().sum()}\n")

数据总行数: 25000
时间范围: 2026-03-07 18:12:00 → 2026-03-10 22:44:51
缺失值:
timestamp        0
temperature_c    0
humidity_pct     0
pressure_hpa     0
dtype: int64



In [6]:
FEATURES=["temperature_c", "humidity_pct", "pressure_hpa"]
data=df[FEATURES].values

In [7]:
scaler=MinMaxScaler()
data_scaled= scaler.fit_transform(data)
target_index=FEATURES.index(TARGET_COL)
target_min=scaler.data_min_[target_index]
target_max=scaler.data_max_[target_index]

In [8]:
def create_sequences(data, window_size, target_col_index,prediction_ahead):    # 创建滑动窗口
    total = len(data) - window_size - prediction_ahead
    X = np.zeros((total, window_size, data.shape[1]), dtype=np.float32)
    y = np.zeros(total, dtype=np.float32)
    for i in range(total):
        X[i] = data[i : i + window_size]
        y[i] = data[i + window_size + prediction_ahead, target_col_index]
    return X, y

In [9]:
X,y=create_sequences(data_scaled, WINDOW_SIZE, target_index, PREDICTION_AHEAD)

In [10]:
split=int(len(X)*0.8)
X_train,X_test=X[:split],X[split:]
y_train,y_test=y[:split],y[split:]

In [11]:
X_train=torch.FloatTensor(X_train)
X_test=torch.FloatTensor(X_test)
y_train=torch.FloatTensor(y_train)
y_test=torch.FloatTensor(y_test)

In [12]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [13]:
train_loader = DataLoader(TimeSeriesDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TimeSeriesDataset(X_test, y_test),
                          batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm=nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.fc=nn.Linear(hidden_size,1)
    
    def forward(self,x):
        out,(hn,cn)=self.lstm(x)
        last_step=out[:,-1,:]
        return self.fc(last_step)  

In [15]:
model = LSTMModel(input_size=3, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS)
model = model.to(device)

TypeError: LSTMModel.__init__() got an unexpected keyword argument 'input_size'